# <font color='#BFD72F' size=6>**0. Imports**</font> <a class="anchor" id="1"></a>

[Back to TOC](#toc)

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

SEED = 42

# <font color='#BFD72F' size=6>**1. Data import**</font> <a class="anchor" id="1"></a>

[Back to TOC](#toc)

In [2]:
train_raw = pd.read_csv("data/raw/train.csv")
test_raw = pd.read_csv("data/raw/test.csv")
clientes_raw = pd.read_csv("data/raw/clientes.csv")

### 1.1 Column-name standardisation

The CSV files are stored as `train_raw`, `test_raw`, and `clientes_raw` for traceability. Standardised copies with lowercase English `snake_case` names become the working datasets used by every subsequent cleaning step.

In [3]:
import re
import unicodedata


english_column_names = {
    "Total a Pagar": "total_payable",
    "Total Bruto": "gross_total",
    "Total Liquido": "net_total",
    "Total Imp_": "total_tax",
    "Total Portes": "shipping_cost",
    "Total Desc_ Global": "global_discount_total",
    "Total Desc_ Linha": "line_discount_total",
    "Custo": "cost",
    "N_ Detalhes": "line_item_count",
}


def to_snake_case(name):
    name = unicodedata.normalize("NFKD", str(name))
    name = name.encode("ascii", "ignore").decode("ascii")
    name = re.sub(r"[^a-z0-9]+", "_", name.lower()).strip("_")
    return re.sub(r"_+", "_", name)


def standardize_columns(df):
    clean_df = df.copy()
    clean_df.columns = [
        english_column_names.get(column, to_snake_case(column))
        for column in clean_df.columns
    ]
    if clean_df.columns.duplicated().any():
        raise ValueError("Column standardisation created duplicate names.")
    return clean_df


In [4]:
train = standardize_columns(train_raw)
test = standardize_columns(test_raw)
clientes = standardize_columns(clientes_raw)

column_changes = pd.DataFrame({
    "original": train_raw.columns,
    "standardized": train.columns,
})

display(column_changes.query("original != standardized"))
print("All standardized schemas have unique column names.")


,original,standardized
0,Total a Pagar,total_payable
1,Total Bruto,gross_total
2,Total Liquido,net_total
3,Total Imp_,total_tax
4,Total Portes,shipping_cost
5,Total Desc_ Global,global_discount_total
6,Total Desc_ Linha,line_discount_total
7,Custo,cost
8,N_ Detalhes,line_item_count
38,ID,id


All standardized schemas have unique column names.


# <font color='#BFD72F' size=6>**2. Cleaning and missing values**</font> <a class="anchor" id="1"></a>

[Back to TOC](#toc)

### 2.1 Almost-empty rows

Exploratory analysis found 65 rows where only `id`, `customer_id`, and the classification target are present. We reproduce that pattern using all remaining columns rather than a manually selected subset.

In [ ]:
preserved_columns = ["id", "customer_id", "repeat_purchase_90d"]
almost_empty_columns = [
    column for column in train.columns
    if column not in preserved_columns
]
missing_pattern = train[almost_empty_columns].isna().all(axis=1)

affected = train.loc[missing_pattern, ["id", "customer_id"]].copy()

# Check whether each customer_id exists in clientes.
customer_id_col = "customer_id"

affected["exists_in_clientes"] = affected["customer_id"].isin(
    clientes[customer_id_col]
)

# Display the results
print("Affected rows:", len(affected))
print("\nCross-check results:")
print(affected["exists_in_clientes"].value_counts(dropna=False))

print("\nCustomers not found in clientes:")
print(
    affected.loc[~affected["exists_in_clientes"]]
    .sort_values("customer_id")
)


### 2.2 Invalid numeric values

Negative amounts, counts, and time intervals are invalid. The exact value `1,000,000,000` appears only in two historical-spend features, so it is treated as an invalid placeholder. Invalid `prior_mean_purchase_value` values are recomputed from total spend and purchase count; the other invalid values become missing values. No rows are removed.


In [6]:
non_negative_columns = [
    "total_payable", "gross_total", "net_total", "total_tax",
    "shipping_cost", "global_discount_total", "line_discount_total",
    "cost", "line_item_count", "invoice_count_current_purchase",
    "days_since_previous_purchase", "customer_tenure_days",
    "prior_purchase_count", "prior_total_spend",
    "prior_mean_purchase_value", "prior_std_purchase_value",
    "prior_min_purchase_value", "prior_max_purchase_value",
    "prior_mean_gap_days", "prior_std_gap_days",
    "purchase_count_30d", "spend_30d", "purchase_count_90d",
    "spend_90d", "purchase_count_180d", "spend_180d",
    "purchase_count_365d", "spend_365d",
]
placeholder_columns = ["prior_mean_purchase_value", "spend_365d"]

invalid_value_report = []
for dataset_name, df in {"train": train, "test": test}.items():
    for column in non_negative_columns:
        count = int(df[column].lt(0).sum())
        if count:
            invalid_value_report.append([dataset_name, column, "negative", count])

    for column in placeholder_columns:
        count = int(df[column].eq(1_000_000_000).sum())
        if count:
            invalid_value_report.append([dataset_name, column, "1,000,000,000 placeholder", count])

invalid_value_report = pd.DataFrame(
    invalid_value_report, columns=["dataset", "column", "issue", "count"]
)
display(invalid_value_report)


,dataset,column,issue,count
0,train,days_since_previous_purchase,negative,16
1,train,prior_mean_purchase_value,negative,16
2,train,prior_mean_gap_days,negative,16
3,train,spend_365d,negative,15
4,train,prior_mean_purchase_value,"1,000,000,000 placeholder",16
5,train,spend_365d,"1,000,000,000 placeholder",15
6,test,days_since_previous_purchase,negative,2
7,test,prior_mean_purchase_value,negative,2
8,test,prior_mean_gap_days,negative,2
9,test,spend_365d,negative,2


In [7]:
def replace_invalid_values(df):
    checked = df.copy()
    invalid_mean_purchase = (
        df["prior_mean_purchase_value"].lt(0)
        | df["prior_mean_purchase_value"].eq(1_000_000_000)
    )
    checked[non_negative_columns] = checked[non_negative_columns].mask(
        checked[non_negative_columns].lt(0)
    )

    for column in placeholder_columns:
        checked.loc[checked[column].eq(1_000_000_000), column] = np.nan

    can_recompute_mean = invalid_mean_purchase & checked["prior_purchase_count"].gt(0)
    checked.loc[can_recompute_mean, "prior_mean_purchase_value"] = (
        checked.loc[can_recompute_mean, "prior_total_spend"]
        / checked.loc[can_recompute_mean, "prior_purchase_count"]
    )

    return checked, int(can_recompute_mean.sum())


train_clean, train_recomputed = replace_invalid_values(train)
test_clean, test_recomputed = replace_invalid_values(test)

print(
    "Recomputed prior mean purchase values:",
    train_recomputed, "in train,", test_recomputed, "in test."
)
print("Other invalid values were replaced by missing values.")


Recomputed prior mean purchase values: 32 in train, 4 in test.
Other invalid values were replaced by missing values.


### 2.3 Categorical text values

We compare categories before and after trimming whitespace and standardising case. No case or spacing variants are found, so the original category labels are kept.


In [8]:
categorical_columns = [
    "payment_type", "salesperson", "commercial_zone",
    "transport_method", "warehouse", "document_series",
]

category_text_report = []
for column in categorical_columns:
    values = pd.concat([train_clean[column], test_clean[column]]).dropna().astype(str)
    normalized = values.str.strip().str.casefold()
    category_text_report.append({
        "column": column,
        "values_with_whitespace": int(values.ne(values.str.strip()).sum()),
        "case_or_spacing_variants": values.nunique() - normalized.nunique(),
    })

category_text_report = pd.DataFrame(category_text_report)
display(category_text_report)
assert category_text_report[["values_with_whitespace", "case_or_spacing_variants"]].eq(0).all().all()


,column,values_with_whitespace,case_or_spacing_variants
0,payment_type,0,0
1,salesperson,0,0
2,commercial_zone,0,0
3,transport_method,0,0
4,warehouse,0,0
5,document_series,0,0


In [ ]:
train_clean = train_clean.loc[~missing_pattern].copy()

print("Rows retained after removing almost-empty rows:", len(train_clean))
print("Rows excluded for broad missingness:", int(missing_pattern.sum()))

### 2.4 Regression-target missingness

In [ ]:
# Create a categorical variable indicating whether the regression target is missing
spend_status = train_clean["spend_next_90d"].isna().map({
    True: "Missing",
    False: "Observed"
})

# Summarise missingness by the classification target
# Rows = repeat_purchase_90d; columns = missingness status
missingness_counts = pd.crosstab(
    train_clean["repeat_purchase_90d"],
    spend_status
)

# Calculate row percentages to compare missingness rates within each class
missingness_percentages = pd.crosstab(
    train_clean["repeat_purchase_90d"],
    spend_status,
    normalize="index"
).mul(100).round(2)

print("Counts:")
print(missingness_counts)

print("\nPercentages within each repeat_purchase_90d class:")
print(missingness_percentages)

In [ ]:

# Focus on repeat purchasers
repeaters = train_clean.loc[
    train_clean["repeat_purchase_90d"] == 1
].copy()

# Identify rows with missing regression targets
missing_target = repeaters["spend_next_90d"].isna()

# Exclude identifiers and target columns from the plots
excluded_cols = [
    "id",
    "customer_id",
    "repeat_purchase_90d",
    "spend_next_90d",
]

numerical_features = [
    col for col in repeaters.select_dtypes(include="number").columns
    if col not in excluded_cols
]

rng = np.random.default_rng(42)

for feature in numerical_features:

    # Separate observed and missing-target groups
    observed = repeaters.loc[~missing_target, feature].dropna()
    missing = repeaters.loc[missing_target, feature].dropna()

    all_values = pd.concat([observed, missing])

    if all_values.empty:
        continue

    # Determine whether a logarithmic scale is appropriate.
    # Use it only when all observed values are strictly positive
    # and the feature has a sufficiently wide range.
    use_log = (
        (all_values > 0).all()
        and all_values.quantile(0.95) / all_values.quantile(0.05) > 20
        and all_values.quantile(0.05) > 0
    )

    # Create vertical jitter to make overlapping observations visible
    observed_y = rng.uniform(-0.15, 0.15, len(observed))
    missing_y = rng.uniform(-0.15, 0.15, len(missing))

    fig, ax = plt.subplots(figsize=(10, 3.5))

    ax.scatter(
        observed, observed_y,
        color="dodgerblue",
        alpha=0.4,
        s=20,
        label="Spending target observed"
    )

    ax.scatter(
        missing, missing_y,
        color="darkorange",
        alpha=0.85,
        s=35,
        label="Spending target missing"
    )

    if use_log:
        # Log scale helps display values spanning several orders of magnitude
        ax.set_xscale("log")
        scale_description = "log scale"

    else:
        # For features with zero/negative values, preserve the original scale.
        # If extreme values exist, limit the visible axis to the 1st–99th
        # percentiles. This changes only the view, not the underlying data.
        lower = all_values.quantile(0.01)
        upper = all_values.quantile(0.99)

        q1 = all_values.quantile(0.25)
        q3 = all_values.quantile(0.75)
        iqr = q3 - q1

        has_extreme_outliers = (
            iqr > 0
            and (
                all_values.min() < q1 - 3 * iqr
                or all_values.max() > q3 + 3 * iqr
            )
        )

        if has_extreme_outliers and lower < upper:
            ax.set_xlim(lower, upper)
            scale_description = "original scale; 1st–99th percentile view"
        else:
            scale_description = "original scale"

    ax.set_title(f"{feature} ({scale_description})")
    ax.set_xlabel(feature)
    ax.set_yticks([])
    ax.set_ylabel("Observations")
    ax.grid(axis="x", alpha=0.25)
    ax.legend()
    plt.tight_layout()
    plt.show()


In [ ]:

# Focus on repeat purchasers
repeaters = train_clean.loc[
    train_clean["repeat_purchase_90d"] == 1
].copy()

# Identify whether the regression target is missing
repeaters["target_status"] = np.where(
    repeaters["spend_next_90d"].isna(),
    "Missing target",
    "Observed target"
)

# Exclude identifiers and target variables
excluded_cols = [
    "id",
    "customer_id",
    "repeat_purchase_90d",
    "spend_next_90d",
    "target_status",
    "purchase_date",
]

# Identify categorical columns automatically
categorical_features = [
    col for col in repeaters.select_dtypes(
        include=["object", "category", "string"]
    ).columns
    if col not in excluded_cols and repeaters[col].nunique(dropna=True) <= 20
]

for feature in categorical_features:

    # Treat missing feature values as a separate category
    data = repeaters[[feature, "target_status"]].copy()
    data[feature] = data[feature].fillna("Missing category").astype(str)

    # Count observations in each category and target-status group
    counts = pd.crosstab(
        data[feature],
        data["target_status"]
    )

    # Ensure both groups appear, even if one has zero observations
    counts = counts.reindex(
        columns=["Observed target", "Missing target"],
        fill_value=0
    )

    # Skip features with no category variation
    if counts.empty:
        continue

    # --------------------------------------------------
    # Plot 1: 100% stacked bars
    # Shows the category distribution within each group
    # --------------------------------------------------
    group_totals = counts.sum(axis=0)
    distribution = counts.div(group_totals.replace(0, np.nan), axis=1) * 100

    ax = distribution.T.plot(
        kind="bar",
        stacked=True,
        figsize=(10, 5),
        width=0.7
    )

    ax.set_title(f"{feature}: Category Distribution by Target Status")
    ax.set_xlabel("Target status")
    ax.set_ylabel("% within target-status group")
    ax.tick_params(axis="x", rotation=0)
    ax.legend(title=feature, bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    plt.show()

    # --------------------------------------------------
    # Plot 2: Missing-target percentage within each category
    # --------------------------------------------------
    category_totals = counts.sum(axis=1)

    missing_percentage = (
        counts["Missing target"] / category_totals.replace(0, np.nan) * 100
    ).sort_values(ascending=False)

    # Exclude categories with no observations
    missing_percentage = missing_percentage.dropna()

    ax = missing_percentage.plot(
        kind="bar",
        figsize=(10, 4),
        color="darkorange"
    )

    ax.set_title(f"{feature}: Percentage Missing Within Each Category")
    ax.set_xlabel(feature)
    ax.set_ylabel("% of repeat purchasers in category with missing target")
    ax.tick_params(axis="x", rotation=45)
    ax.axhline(
        100 * repeaters["spend_next_90d"].isna().mean(),
        linestyle="--",
        color="black",
        label="Overall missing-target rate"
    )
    ax.legend()
    plt.tight_layout()
    plt.show()


### 2.5 Missing-value treatment decision

- **Broad missingness:** exclude the 65 training rows with a simultaneous, broad predictor-missingness pattern from model fitting. There is no equivalent complete pattern in `test.csv`, and imputation would invent most of a purchase record. Keep the raw rows for audit only.
- **Predictor missingness:** retain the remaining missing feature values until the modelling pipeline. Numeric imputation, categorical `Missing` labels, and missingness indicators must be fitted on each training fold only to avoid leakage.
- **Targets:** never impute either target. The classification target is complete. Train the spend regressor only on repeat purchasers with an observed `spend_next_90d` value.
- **Clientes:** if the optional customer table is merged later, drop `customer_type` because it is entirely missing; handle the other customer fields within the same fold-safe preprocessing pipeline.

In [ ]:
assert train_clean["repeat_purchase_90d"].notna().all()

eligible_repeaters = train_clean.loc[
    train_clean["repeat_purchase_90d"].eq(1)
].copy()
observed_regression_target = eligible_repeaters["spend_next_90d"].notna()

print("Classification training rows:", len(train_clean))
print("Regression training rows (observed spend only):", int(observed_regression_target.sum()))
print(
    "Repeat purchasers without an observed spend target:",
    int((~observed_regression_target).sum())
)

### 2.6 Train/test feature compatibility

The two targets are expected to exist only in `train`. For every predictor shared by train and test, the data type must have the same broad meaning (numeric or text). Exact pandas dtypes may differ because missing numeric values are stored as floats.


In [ ]:
target_columns = ["repeat_purchase_90d", "spend_next_90d"]
train_feature_columns = set(train_clean.columns) - set(target_columns)
test_feature_columns = set(test_clean.columns)

print("Features missing from test:", sorted(train_feature_columns - test_feature_columns))
print("Unexpected features in test:", sorted(test_feature_columns - train_feature_columns))

def dtype_family(series):
    return "numeric" if pd.api.types.is_numeric_dtype(series) else "text"

type_report = pd.DataFrame({
    "train_dtype": train_clean.dtypes,
    "test_dtype": test_clean.dtypes,
}).loc[sorted(train_feature_columns & test_feature_columns)]
type_report["train_family"] = [dtype_family(train_clean[column]) for column in type_report.index]
type_report["test_family"] = [dtype_family(test_clean[column]) for column in type_report.index]

display(type_report.query("train_dtype != test_dtype"))
assert train_feature_columns == test_feature_columns
assert type_report["train_family"].equals(type_report["test_family"])


### 2.7 Test categories unseen in training

Some valid test categories do not appear in training. They are not renamed: the later encoder must handle unknown categories safely.


In [ ]:
unseen_category_report = []
for column in categorical_columns:
    train_categories = set(train_clean[column].dropna().astype(str))
    test_categories = set(test_clean[column].dropna().astype(str))
    unseen_in_train = sorted(test_categories - train_categories)
    unseen_category_report.append({
        "column": column,
        "test_categories_unseen_in_train": ", ".join(unseen_in_train) or "None",
    })

unseen_category_report = pd.DataFrame(unseen_category_report)
display(unseen_category_report.query("test_categories_unseen_in_train != 'None'"))


# <font color='#BFD72F' size=6>**3. Date validation**</font> <a class="anchor" id="1"></a>

- 3.1 **Date validity:** Count missing and unparseable dates in both datasets. Avoid crashes when it contains no valid dates.
- 3.2 **Temporal coverage:** Report the minimum and maximum valid dates for each dataset, with consistent formatting.
- 3.3 **Train/test chronology:** Check whether the training period ends before the test period begins. Report inconsistencies as a warning.
- 3.4 **Temporal distribution:** Check whether dates are concentrated in particular periods and whether there are unexpected gaps, if the data is expected to cover a continuous timeline.

[Back to TOC](#toc)

In [ ]:

def validate_purchase_dates(train, test):
    """Validate purchase dates and compare train/test time periods."""

    datasets = {"train": train.copy(), "test": test.copy()}
    date_col = "purchase_date"
    date_ranges = {}

    # Step 1: Check that both datasets contain the date column.
    for name, df in datasets.items():
        if date_col not in df.columns:
            raise KeyError(f"'{date_col}' is missing from {name}.")

    # Step 2: Parse dates and report invalid or missing values.
    for name, df in datasets.items():
        # Convert invalid date strings to NaT instead of raising errors.
        df[date_col] = pd.to_datetime(
            df[date_col],
            errors="coerce"
        )

        valid_dates = df[date_col].dropna()
        invalid_count = df[date_col].isna().sum()

        print(f"\n--- {name.upper()} DATE VALIDATION ---")
        print(f"Total observations: {len(df)}")
        print(f"Missing/unparseable dates: {invalid_count}")

        # Step 3: Calculate the date interval safely.
        if valid_dates.empty:
            print("No valid dates available.")
            date_ranges[name] = None
            continue

        earliest = valid_dates.min()
        latest = valid_dates.max()

        date_ranges[name] = (earliest, latest)

        print(f"Earliest date: {earliest}")
        print(f"Latest date:   {latest}")

        monthly_counts = valid_dates.dt.to_period("M").value_counts().sort_index()
        full_period = pd.period_range(
            earliest.to_period("M"), latest.to_period("M"), freq="M"
        )
        missing_months = full_period.difference(monthly_counts.index)
        print(f"Months without purchases: {len(missing_months)}")

    # Step 4: Compare the temporal coverage of both datasets.
    train_range = date_ranges["train"]
    test_range = date_ranges["test"]

    if train_range is None or test_range is None:
        print("\nCannot compare periods: at least one dataset has no valid dates.")
        return datasets["train"], datasets["test"]

    train_start, train_end = train_range
    test_start, test_end = test_range

    print("\n--- TRAIN/TEST CHRONOLOGY ---")

    if train_end < test_start:
        print("PASS: Training dates precede test dates.")
    elif test_end < train_start:
        print("WARNING: Test period precedes training period.")
    else:
        print("WARNING: Train/test date ranges overlap.")

        overlap_start = max(train_start, test_start)
        overlap_end = min(train_end, test_end)

        print(f"Overlapping interval: {overlap_start} to {overlap_end}")

    return datasets["train"], datasets["test"]


train_clean, test_clean = validate_purchase_dates(train_clean, test_clean)

# <font color='#BFD72F' size=6>**4. Duplicates**</font> <a class="anchor" id="1"></a>

- 4.1 Detect exact duplicate rows in each dataset.
- 4.2 Report how many duplicate rows exist and display examples.
- 4.3 **Optionally** remove exact duplicates, keeping the first occurrence.
- 4.4 Check for duplicates across train and test, which can reveal possible data leakage.
- 4.5 **Optionally** check duplicates using specified identifier columns.

[Back to TOC](#toc)

In [14]:
def validate_and_handle_duplicates(
    train,
    test,
    key_cols=None,
    remove_exact=True,
    remove_key_duplicates=False,
    show_examples=True,
    max_examples=10,
):
    """
    Detect and optionally remove duplicates in train and test datasets.

    Parameters
    ----------
    train, test : pandas.DataFrame
        Training and test datasets.
    key_cols : list[str] or None
        Columns that should uniquely identify an observation.
        Examples: ["transaction_id"] or ["customer_id", "purchase_date"].
    remove_exact : bool
        Remove exact duplicate rows within each dataset, keeping the first.
    remove_key_duplicates : bool
        Remove duplicate observations based on key_cols, keeping the first.
        Use only when those columns genuinely identify unique observations.
    show_examples : bool
        Display examples of duplicate rows.
    max_examples : int
        Maximum number of duplicate examples to display.

    Returns
    -------
    train_clean, test_clean : pandas.DataFrame
        Cleaned copies of the input datasets.
    report : dict
        Duplicate counts and cross-dataset diagnostics.
    """

    datasets = {"train": train, "test": test}
    cleaned = {}
    report = {}

    # Step 1: Validate the inputs and key columns.
    if not isinstance(train, pd.DataFrame) or not isinstance(
        test, pd.DataFrame
    ):
        raise TypeError("train and test must be pandas DataFrames.")

    if train.columns.has_duplicates or test.columns.has_duplicates:
        raise ValueError("Duplicate column names must be resolved first.")

    if key_cols is not None:
        if isinstance(key_cols, str):
            key_cols = [key_cols]
        else:
            key_cols = list(key_cols)

        if not key_cols:
            raise ValueError("key_cols cannot be an empty list.")

        for name, df in datasets.items():
            missing_cols = set(key_cols) - set(df.columns)
            if missing_cols:
                raise ValueError(
                    f"{name} is missing key columns: {sorted(missing_cols)}"
                )

    # Step 2: Inspect and optionally remove duplicates within each dataset.
    for name, df in datasets.items():
        result = df.copy()

        # Detect exact duplicate rows, excluding the first occurrence.
        exact_mask = result.duplicated(keep="first")
        exact_count = int(exact_mask.sum())

        print(f"\n--- {name.upper()} DUPLICATE VALIDATION ---")
        print(f"Original rows: {len(result)}")
        print(f"Exact duplicate rows: {exact_count}")

        if show_examples and exact_count > 0:
            print("Examples of exact duplicates:")
            display(result.loc[exact_mask].head(max_examples))

        # Optionally remove exact duplicate rows.
        if remove_exact:
            result = result.drop_duplicates(keep="first")

        key_report = None

        # Step 3: Check duplicates based on the chosen observation key.
        if key_cols is not None:
            # Rows with incomplete keys cannot be reliably matched.
            complete_key_mask = result[key_cols].notna().all(axis=1)
            incomplete_key_count = int((~complete_key_mask).sum())

            # Only evaluate uniqueness among rows with complete keys.
            complete_rows = result.loc[complete_key_mask]
            key_duplicate_mask = complete_rows.duplicated(
                subset=key_cols,
                keep="first",
            )
            key_duplicate_count = int(key_duplicate_mask.sum())

            print(f"Rows with incomplete keys: {incomplete_key_count}")
            print(f"Duplicate key rows: {key_duplicate_count}")

            if show_examples and key_duplicate_count > 0:
                print(f"Examples of duplicate keys ({key_cols}):")
                display(
                    complete_rows.loc[key_duplicate_mask]
                    .head(max_examples)
                )

            # Remove duplicate keys only when explicitly requested.
            if remove_key_duplicates:
                duplicate_index = complete_rows.index[key_duplicate_mask]
                result = result.drop(index=duplicate_index)

            key_report = {
                "key_columns": key_cols,
                "incomplete_key_rows": incomplete_key_count,
                "duplicate_key_rows": key_duplicate_count,
            }

        cleaned[name] = result

        report[name] = {
            "original_rows": len(df),
            "exact_duplicate_rows": exact_count,
            "rows_after_exact_deduplication": (
                len(df.drop_duplicates()) if remove_exact else len(df)
            ),
            "final_rows": len(result),
            "key_diagnostics": key_report,
        }

        print(f"Rows after handling duplicates: {len(result)}")

    # Step 4: Check for exact row matches between train and test.
    # This is diagnostic only: do not automatically remove test rows.
    train_clean = cleaned["train"]
    test_clean = cleaned["test"]

    comparison_cols = list(
        col for col in train_clean.columns
        if col in test_clean.columns and col != "id"
    )

    if not comparison_cols:
        print("\nCannot compare train/test rows: no shared columns.")
        cross_exact_count = None
    else:
        # Hash corresponding shared-column values consistently.
        # Hashes allow comparison without a potentially large DataFrame merge.
        train_hashes = pd.util.hash_pandas_object(
            train_clean[comparison_cols], index=False
        )
        test_hashes = pd.util.hash_pandas_object(
            test_clean[comparison_cols], index=False
        )

        train_hash_set = set(train_hashes)
        cross_exact_count = int(test_hashes.isin(train_hash_set).sum())

        print("\n--- TRAIN/TEST OVERLAP ---")
        print(
            "Test rows matching a train row on shared non-ID columns:",
            cross_exact_count,
        )

        if set(train_clean.columns) != set(test_clean.columns):
            print(
                "WARNING: Cross-dataset comparison uses shared columns only; "
                "matches are not necessarily identical full observations."
            )
        else:
            print(
                "Note: Matching rows are flagged for investigation, "
                "not automatically removed."
            )

    report["cross_dataset"] = {
        "comparison_columns": comparison_cols,
        "matching_test_rows": cross_exact_count,
    }

    return train_clean, test_clean, report


In [ ]:
train_clean, test_clean, duplicate_report = (
    validate_and_handle_duplicates(
        train_clean,
        test_clean,
        key_cols=["id"],
        remove_exact=True,
        remove_key_duplicates=False,
    )
)

train_for_classification = train_clean.copy()
repeaters = train_for_classification.loc[
    train_for_classification["repeat_purchase_90d"].eq(1)
].copy()
train_for_regression = repeaters.dropna(subset=["spend_next_90d"]).copy()

print("Final classification rows:", len(train_for_classification))
print("Final regression rows:", len(train_for_regression))

# <font color='#BFD72F' size=6>**5. From diagnosis to pipeline rules**</font> <a class="anchor" id="rules"></a>

Sections 1–4 diagnosed the data. From here on the diagnosis becomes the code in `src/preprocessing.py`, driven by `config.yaml`. This notebook imports those functions instead of redefining them, so the notebook and `main.py` always run exactly the same code.

Every step is one of three kinds, and the kind decides where it may run:

| Kind | In this project | Learns from data? | Where it runs |
|---|---|---|---|
| **Stateless rule** | renames, negatives/placeholders → `NaN`, identity recomputation, structural fill, flags, redundant columns (`clean_dataset`) | No | train.csv **and** test.csv |
| **Training-data decision** | almost-empty rows, duplicates (`drop_training_rows`), regression rows (`regression.select_training_rows`) | No, but **removes rows** | labelled training file only, before the dev/test split |
| **Fitted step** | imputation, encoding, scaling (`build_preprocessor`) | **Yes** | inside the model `Pipeline`, fitted on training rows only |

[Back to TOC](#toc)

In [ ]:
import yaml

from src.preprocessing import (
    load_data, clean_dataset, drop_training_rows, split_dev_test, build_preprocessor,
    task_preprocessing_config,
)
import classification.preprocessing as classification_prep
import regression.preprocessing as regression_prep

with open("config.yaml", encoding="utf-8") as f:
    config = yaml.safe_load(f)
data_cfg, diag_cfg, prep_cfg = config["data"], config["diagnostics"], config["preprocessing"]

### 5.1 Values that can be recovered from identities

Several history columns are linked by exact definitions. Wherever one term is missing (originally, or after the negative/placeholder → `NaN` step), it can be recomputed **from the same row**. Nothing is learned from other rows, so this is a stateless rule and runs on test.csv too. The rules live in `config.yaml` → `diagnostics.recompute_rules`. Each one is kept only if it holds on almost every row where all its terms are observed and valid:

In [ ]:
evidence = train.loc[~missing_pattern].copy()
evidence[non_negative_columns] = evidence[non_negative_columns].mask(evidence[non_negative_columns].lt(0))
for column in placeholder_columns:
    evidence[column] = evidence[column].mask(evidence[column].eq(1_000_000_000))

identity_report = []
for rule in diag_cfg["recompute_rules"]:
    rows = evidence.query(rule["when"])
    expected = rows.eval(rule["expression"])
    observed = rows[rule["column"]]
    both = expected.notna() & observed.notna()
    holds = np.isclose(expected[both], observed[both], rtol=1e-3, atol=0.6)
    identity_report.append({
        "column": rule["column"],
        "identity": f"{rule['expression']}  (when {rule['when']})",
        "rows_checked": int(both.sum()),
        "identity_holds_%": round(100 * holds.mean(), 2),
        "NaN_it_can_fill": int((observed.isna() & expected.notna()).sum()),
    })
display(pd.DataFrame(identity_report))

**Redundant and non-informative columns** (`diagnostics.redundant_columns`, `data.drop_columns`):

In [ ]:
gross_identity = np.isclose(
    evidence["gross_total"],
    evidence["net_total"] + evidence["line_discount_total"] + evidence["global_discount_total"],
    atol=0.05,
)
print(f"gross_total = net_total + line + global discount: holds for {100 * gross_identity.mean():.1f}% of rows")
print(f"corr(total_payable, net_total) = {evidence['total_payable'].corr(evidence['net_total']):.3f}")
print(f"corr(random_noise, repeat_purchase_90d) = {evidence['random_noise'].corr(evidence['repeat_purchase_90d']):.3f}")

- `gross_total` is an exact sum of three other columns, so a linear model gets the same information twice → dropped.
- `total_payable` and `net_total` are correlated at 0.996 → `total_payable` dropped.
- `random_noise` has no relationship with the target and, as its name says, is noise → dropped.

### 5.2 Structural missingness: values that cannot exist

Missing history values are concentrated where the value is undefined: a first purchase has no previous purchase, a standard deviation needs at least two values, and a standard deviation of gaps needs at least two gaps. Missing rate (%) by number of prior purchases, on the raw training rows:

In [ ]:
history_columns = [
    "days_since_previous_purchase", "prior_mean_purchase_value", "prior_min_purchase_value",
    "prior_max_purchase_value", "prior_std_purchase_value", "prior_mean_gap_days", "prior_std_gap_days",
]
history = train.loc[~missing_pattern]
prior_purchases = history["prior_purchase_count"].clip(upper=3).astype(int).astype(str).replace({"3": "3+"})
structural_table = history[history_columns].isna().groupby(prior_purchases).mean().mul(100).round(1).T
structural_table.columns.name = "prior_purchase_count"
display(structural_table)

Each column is 100% missing exactly where it is undefined. The few gaps left at 1/2/3+ prior purchases come from the corrupted values and are mostly recovered by the identities in 5.1. So these gaps are **not** MCAR/MAR/MNAR in the usual sense: they are *not applicable*. A median would invent a purchase history for a first-time customer, so instead:

- `diagnostics.structural_fill` sets them to **0** (no previous spend, no spread, no gap). This is a rule from the column definitions, not a fitted statistic.
- the flags `is_first_purchase` and `has_gap_history` let a model tell "0 because undefined" apart from a real 0.

### 5.3 Applying the rules

`clean_dataset()` runs every rule above on train.csv and test.csv and reports every value it changed. `drop_training_rows()` is called on the training file only.

In [ ]:
train_rules, cleaning_report = clean_dataset(train_raw, diag_cfg)
test_rules, _ = clean_dataset(test_raw, diag_cfg)
assert len(train_rules) == len(train_raw) and len(test_rules) == len(test_raw), "clean_dataset must keep every row"
display(cleaning_report)

train_rules, row_report = drop_training_rows(train_rules, diag_cfg, data_cfg["id_column"])
display(row_report)
print(f"train.csv: {len(train_raw)} -> {len(train_rules)} training rows | test.csv: {len(test_raw)} -> {len(test_rules)} rows (none dropped)")

# <font color='#BFD72F' size=6>**6. Missingness mechanism of the remaining gaps**</font> <a class="anchor" id="mechanism"></a>

After the rules, only real gaps are left. For each one we test whether *being missing* is related to the target or to any other feature:

- **target:** χ² test of missing/observed × `repeat_purchase_90d`;
- **numeric features:** Mann-Whitney U, missing rows vs observed rows;
- **categorical features:** χ² test, with categories under 30 rows grouped as `rare` so that sparse cells don't produce false signals.

Significance uses a Bonferroni threshold (0.05 / number of tests per column). No association → **MCAR** → a simple median/mode loses nothing. Association → **MAR/MNAR** → impute **and** add a `_was_missing` flag.

[Back to TOC](#toc)

In [ ]:
from scipy import stats


def missingness_tests(df, columns, numeric, categorical, target, min_category_rows=30):
    rows = []
    for column in columns:
        missing = df[column].isna()
        target_p = stats.chi2_contingency(pd.crosstab(missing, df[target]))[1]

        p_values = {}
        for other in numeric:
            if other != column and df.loc[missing, other].notna().sum() > 5:
                p_values[other] = stats.mannwhitneyu(
                    df.loc[missing, other].dropna(), df.loc[~missing, other].dropna()
                ).pvalue
        for other in categorical:
            if other != column:
                values = df[other].fillna("missing")
                counts = values.value_counts()
                values = values.where(values.map(counts).ge(min_category_rows), "rare")
                p_values[other] = stats.chi2_contingency(pd.crosstab(missing, values))[1]

        threshold = 0.05 / (len(p_values) + 1)
        related = [other for other, p in sorted(p_values.items(), key=lambda kv: kv[1]) if p < threshold]
        rows.append({
            "column": column,
            "missing_%": round(100 * missing.mean(), 2),
            "target_rate_missing": round(df.loc[missing, target].mean(), 3),
            "target_rate_observed": round(df.loc[~missing, target].mean(), 3),
            "target_p": round(target_p, 3),
            "related_features": ", ".join(related[:4]) or "none",
            "verdict": "MCAR" if target_p >= threshold and not related else "MAR/MNAR -> flag",
        })
    return pd.DataFrame(rows)


model_columns = prep_cfg["numeric_features"] + prep_cfg["categorical_features"]
remaining_gaps = [c for c in model_columns if train_rules[c].isna().any()]
mechanism_table = missingness_tests(
    train_rules, remaining_gaps, prep_cfg["numeric_features"], prep_cfg["categorical_features"],
    config["classification"]["target"],
)
display(mechanism_table)

`spend_365d` shows up as related to tenure and history. Is that a property of the raw data, or a result of our own identity fill (5.1 fills it only for customers with ≤ 365 days of tenure)?

In [ ]:
raw_spend_missing = history["spend_365d"].isna()
print("Raw data, tenure missing vs observed, Mann-Whitney p =",
      round(stats.mannwhitneyu(history.loc[raw_spend_missing, "customer_tenure_days"],
                               history.loc[~raw_spend_missing, "customer_tenure_days"]).pvalue, 3))
still_missing = train_rules["spend_365d"].isna()
print("After the rules, tenure of rows still missing: min",
      int(train_rules.loc[still_missing, "customer_tenure_days"].min()), "days")

**Verdict table** (stored in `config.yaml` → `preprocessing`):

| Column | Missing after rules | Mechanism | Treatment (fitted on training rows only) |
|---|---|---|---|
| `days_since_previous_purchase`, `prior_mean_gap_days` | 8 rows (0.1%) | MCAR: no relation to target or features | median |
| `spend_365d` | ~2.4% | MCAR in the raw data, but after the identity fill only long-tenure customers stay missing → related to tenure/history | median **+ `spend_365d_was_missing`** (`mnar_indicator_sources`) |
| `payment_type`, `salesperson`, `transport_method`, `warehouse` | ~3% each, never on the same rows | MCAR: no relation to target or features once sparse categories are grouped | most frequent |
| history columns, first/early purchases | structural | not applicable (5.2) | 0 + `is_first_purchase` / `has_gap_history` |
| `spend_next_90d` (target) | all non-repeaters, 5.4% of repeaters | target, never imputed | regression trains on observed rows only |

An automatic `SimpleImputer(add_indicator=True)` would flag *every* column with gaps, including the MCAR ones, where a flag is only noise. We flag only where the evidence says the gap carries information.

# <font color='#BFD72F' size=6>**7. Features, targets and the locked test set**</font> <a class="anchor" id="split"></a>

[Back to TOC](#toc)

In [ ]:
date_col = data_cfg["date_column"]
print(f"train.csv: {train_rules[date_col].min():%Y-%m-%d} -> {train_rules[date_col].max():%Y-%m-%d}")
print(f"test.csv:  {test_rules[date_col].min():%Y-%m-%d} -> {test_rules[date_col].max():%Y-%m-%d}")
shared = set(test_rules["customer_id"]) & set(train_rules["customer_id"])
print(f"{train_rules['customer_id'].nunique()} customers make {len(train_rules)} training purchases; "
      f"{len(shared)} of the {test_rules['customer_id'].nunique()} test customers also appear in train")

test.csv lies **strictly after** train.csv in time, and most of its customers are already in train. A random split would let the model learn from a customer's *later* purchases and then be scored on their earlier ones, which is the opposite of what Kaggle asks. So the locked test set is the **latest 20% of training purchases** (`config.yaml` → `test_set.strategy: temporal`). It is never used to fit, tune, compare or choose anything. Development purchases from the 90 days just before it are left out (`test_set.gap_days`), because their 90-day labels are measured inside the test period.

`split_features_target()` returns `(X, y, extras)`:
- `extras` keeps `id`, `customer_id` and `purchase_date` row-aligned beside `X` (for the split and the submission file), never as model inputs;
- **both** targets are removed from `X`, so the classifier never sees `spend_next_90d` and the regressor never sees `repeat_purchase_90d`;
- `y` is `None` on test.csv: nothing downstream needs the label.

In [ ]:
splits = {}
for task, task_prep in {"classification": classification_prep, "regression": regression_prep}.items():
    task_rows = task_prep.select_training_rows(train_rules, config)
    X, y, extras = task_prep.prepare_features(task_rows, config)
    X_dev, X_test, y_dev, y_test, extras_dev, extras_test = split_dev_test(X, y, extras, config["test_set"], date_col)
    splits[task] = (X_dev, X_test, y_dev, y_test, extras_dev, extras_test)
    print(f"{task:>14}: development {len(X_dev)} rows ({extras_dev[date_col].min():%Y-%m} -> {extras_dev[date_col].max():%Y-%m})"
          f" | locked test {len(X_test)} rows ({extras_test[date_col].min():%Y-%m} -> {extras_test[date_col].max():%Y-%m})")

X_kaggle, y_kaggle, extras_kaggle = classification_prep.prepare_features(test_rules, config)
print(f"{'test.csv':>14}: {len(X_kaggle)} rows, y is {y_kaggle}")

# <font color='#BFD72F' size=6>**8. The recipe: `build_preprocessor()`**</font> <a class="anchor" id="recipe"></a>

One `ColumnTransformer`, built entirely from `config.yaml` → `preprocessing`:

| Group | Steps |
|---|---|
| numeric features | median imputer → scaler (standard for classification, min-max for regression, chosen in section 9) |
| categorical features | most-frequent imputer → target encoder (`handle_unknown`: unseen categories get the overall target mean) |
| flags | `is_first_purchase`, `has_gap_history`, `spend_365d_was_missing` passed through |

The shared default in `config.yaml` → `preprocessing` was a first choice with a reason: target encoding gives one column per feature (`transport_method` has 80 levels), and robust scaling uses median/IQR, so the skewed amounts with extreme values don't set the scale. Section 9 compares every encoder × scaler pair, and each task stores its winner in `config.yaml` → `<task>.preprocessing`; `task_preprocessing_config()` merges the two.

**Nothing is fitted until `fit` is called on training rows.** Below, the recipe is fitted on the development rows of the classification task only, then applied unchanged to the locked test set and to test.csv.

[Back to TOC](#toc)

In [ ]:
X_dev, X_test, y_dev, y_test, _, _ = splits["classification"]
prep = build_preprocessor(task_preprocessing_config(config, "classification")).fit(X_dev, y_dev)

print("Model-ready columns:", len(prep.get_feature_names_out()))
print(list(prep.get_feature_names_out()))
for name, X_part in {"development": X_dev, "locked test": X_test, "test.csv": X_kaggle}.items():
    matrix = prep.transform(X_part)
    print(f"{name:>12}: shape {matrix.shape}, NaN left {int(np.isnan(matrix).sum())}")

**What was learned, and from which rows:** the medians come from the development rows only, so the locked test set and test.csv never influenced them.

In [ ]:
numeric_imputer = prep.named_transformers_["numeric"].named_steps["impute"]
learned = pd.Series(numeric_imputer.statistics_, index=prep_cfg["numeric_features"])
display(learned[["days_since_previous_purchase", "prior_mean_gap_days", "spend_365d"]].to_frame("median learned on development rows"))

**Unseen categories don't crash anything.** test.csv contains categories that never appear in training (section 2.7). A brand-new record with invented values:

In [ ]:
new_purchase = X_kaggle.head(1).copy()
new_purchase[prep_cfg["categorical_features"]] = "never seen before"
encoded = pd.DataFrame(prep.transform(new_purchase), columns=prep.get_feature_names_out())
display(encoded.filter(like="categorical__").round(3))
print("Overall target mean on development rows:", round(y_dev.mean(), 3))

# <font color='#BFD72F' size=6>**9. Choosing the encoder × scaler pair**</font> <a class="anchor" id="encoder-scaler"></a>

4 encoders × 4 scalers = 16 recipes. Instead of picking one by convention, every pair is scored with a real model, and the data decides.

- **Which model:** a scaling-sensitive linear model (`classification.comparison_model`: logistic regression; `regression.comparison_model`: ridge). Trees split on thresholds and don't care about scaling, so they couldn't tell the scalers apart.
- **Which rows:** the development set only. The locked test set plays no part in any choice.
- **Which folds:** 5 **expanding-window temporal folds**. Each fold trains on the past and validates on the next block of purchases, like test.csv. Training rows from the 90 days just before a validation block are left out (`validation.gap_days`), because their 90-day labels are measured inside that block.
- **Same folds for every pair**, so two recipes can be compared fold by fold (a *paired* comparison), not only by their means.

[Back to TOC](#toc)

In [ ]:
from src.evaluate import (
    temporal_folds, make_pipeline, cross_validate_pipeline, compare_encoders_scalers, paired_comparison,
)
import classification.model as classification_model
import regression.model as regression_model

TASK_MODELS = {"classification": classification_model, "regression": regression_model}
ENCODERS = ["onehot", "ordinal", "count", "target"]
SCALERS = ["none", "standard", "minmax", "robust"]

task_folds = {}
for task, (X_dev, X_test, y_dev, y_test, extras_dev, extras_test) in splits.items():
    task_folds[task] = temporal_folds(extras_dev[date_col], config["validation"]["n_splits"], config["validation"]["gap_days"])
    display(pd.DataFrame([
        {"task": task, "fold": k,
         "train_rows": len(tr), "train_until": f"{extras_dev[date_col].iloc[tr].max():%Y-%m-%d}",
         "validation_rows": len(va), "validation_from": f"{extras_dev[date_col].iloc[va].min():%Y-%m-%d}",
         "validation_until": f"{extras_dev[date_col].iloc[va].max():%Y-%m-%d}"}
        for k, (tr, va) in enumerate(task_folds[task])
    ]))

In [ ]:
grid_rankings, grid_scores = {}, {}
for task in ["classification", "regression"]:
    X_dev, _, y_dev, _, _, _ = splits[task]
    task_cfg = config[task]
    grid_rankings[task], grid_scores[task] = compare_encoders_scalers(
        prep_cfg, lambda: TASK_MODELS[task].build_model(task_cfg["comparison_model"]),
        X_dev, y_dev, task_folds[task], task, task_cfg["metric"], ENCODERS, SCALERS,
    )
    print(f"\n{task}: validation {task_cfg['metric']} over {len(task_folds[task])} temporal folds")
    display(grid_rankings[task].round(4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, task, cmap in zip(axes, ["classification", "regression"], ["Blues", "Blues_r"]):
    metric = config[task]["metric"]
    pivot = grid_rankings[task].pivot(index="encoder", columns="scaler", values=f"mean_{metric}").loc[ENCODERS, SCALERS]
    image = ax.imshow(pivot.values, cmap=cmap)
    ax.set_xticks(range(len(SCALERS)), SCALERS)
    ax.set_yticks(range(len(ENCODERS)), ENCODERS)
    for i in range(len(ENCODERS)):
        for j in range(len(SCALERS)):
            ax.text(j, i, f"{pivot.values[i, j]:.3f}" if metric == "accuracy" else f"{pivot.values[i, j]:.0f}",
                    ha="center", va="center", fontsize=9)
    ax.set_title(f"{task}: mean validation {metric}" + (" (lower is better)" if metric == "rmse" else ""))
    fig.colorbar(image, ax=ax, shrink=0.85)
plt.tight_layout()
plt.show()

**Is the winner really better, or is it the luck of the folds?** Same folds → subtract fold by fold. A mean advantage of more than 2 standard errors is a real edge. Within 2 SE, either pair is defensible, and the choice needs another reason.

In [ ]:
pairs_to_check = {
    "classification": [(("target", "standard"), ("target", "robust")), (("target", "standard"), ("onehot", "robust"))],
    "regression": [(("ordinal", "minmax"), ("target", "minmax")), (("target", "minmax"), ("target", "robust"))],
}
rows = []
for task, pairs in pairs_to_check.items():
    higher_is_better = config[task]["metric"] != "rmse"
    for a, b in pairs:
        result = paired_comparison(grid_scores[task][a], grid_scores[task][b], higher_is_better)
        rows.append({"task": task, "A": " + ".join(a), "B": " + ".join(b), **result})
paired_table = pd.DataFrame(rows)
display(paired_table.round(4))

**Decision** (stored in `config.yaml` → `classification.preprocessing` / `regression.preprocessing`):

- **Classification → target encoding + standard scaling.** Target encoding takes the top 3 places, and count encoding takes the bottom 4: equal counts collide, and frequency says little about repurchase. *Standard* and *robust* are tied (+0.0005 accuracy, SE 0.0006), so the grid picks the top one, but the gap is not decisive. Scaling clearly matters: `target + none` drops to 0.696.
- **Regression → target encoding + min-max scaling.** Min-max is best with every encoder. With ridge, the scale decides how strongly each coefficient is penalised, so this is mostly a regularisation effect. `ordinal + minmax` ranks first, but it is tied with `target + minmax` (3 RMSE, SE 19). The tie goes to target encoding, because ordinal encoding would impose an order on unordered categories (salesperson, transport method, …). `target + minmax` beats `target + robust` (the first choice in section 8) by ≈ 147 RMSE and on all 5 folds, but that is 1.7 SE, still under the 2 SE bar.

These pairs were chosen with the linear models. In section 10 the tree models use the same recipe; scaling doesn't change their splits.

# <font color='#BFD72F' size=6>**10. Recipe + model = one Pipeline**</font> <a class="anchor" id="pipeline"></a>

`make_pipeline()` puts `build_preprocessor()` and the model into one sklearn `Pipeline`. In every fold, `fit` learns the medians, the target-encoding means, the scaling statistics **and** the model from that fold's training rows only. `predict` then applies all of it unchanged to the validation block.

Candidates (`config.yaml` → `<task>.candidate_models`), each with its task's chosen recipe:

| | Classification | Regression |
|---|---|---|
| floor to beat | `DummyClassifier` (most frequent class) | `DummyRegressor` (mean) |
| linear | logistic regression | ridge |
| single tree | decision tree, depth 5 | decision tree, depth 5 |
| ensemble | random forest, 300 trees, ≥ 5 rows per leaf | random forest, 300 trees, ≥ 5 rows per leaf |

Hyperparameters are untouched defaults or simple limits for now; tuning comes later. **Train** vs **validation** scores show over- and under-fitting.

[Back to TOC](#toc)

In [ ]:
model_results, model_fold_scores = {}, {}
for task in ["classification", "regression"]:
    X_dev, _, y_dev, _, _, _ = splits[task]
    recipe = task_preprocessing_config(config, task)
    metric = config[task]["metric"]
    rows, model_fold_scores[task] = {}, {}
    for model_cfg in config[task]["candidate_models"]:
        pipe = make_pipeline(recipe, TASK_MODELS[task].build_model(model_cfg))
        scores = cross_validate_pipeline(pipe, X_dev, y_dev, task_folds[task], task)
        rows[model_cfg["type"]] = scores.mean()
        model_fold_scores[task][model_cfg["type"]] = scores[f"validation_{metric}"].to_numpy()
    model_results[task] = pd.DataFrame(rows).T
    print(f"\n{task} ({recipe['encoder']} + {recipe['scaler']}): mean over {len(task_folds[task])} temporal folds")
    display(model_results[task].round(4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
for ax, task in zip(axes, ["classification", "regression"]):
    metric = config[task]["metric"]
    table = model_results[task][[f"train_{metric}", f"validation_{metric}"]]
    positions = np.arange(len(table))
    ax.bar(positions - 0.2, table[f"train_{metric}"], width=0.4, color="#8E9196", label="train")
    ax.bar(positions + 0.2, table[f"validation_{metric}"], width=0.4, color="dodgerblue", label="validation")
    ax.set_xticks(positions, table.index, rotation=15)
    ax.set_title(f"{task}: {metric}" + (" (lower is better)" if metric == "rmse" else ""))
    ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
rows = []
for task, comparisons in {
    "classification": [("logistic_regression", "dummy"), ("logistic_regression", "random_forest")],
    "regression": [("random_forest", "dummy"), ("random_forest", "ridge")],
}.items():
    higher_is_better = config[task]["metric"] != "rmse"
    for a, b in comparisons:
        result = paired_comparison(model_fold_scores[task][a], model_fold_scores[task][b], higher_is_better)
        rows.append({"task": task, "A": a, "B": b, **result})
display(pd.DataFrame(rows).round(4))

**Decision** (stored in `config.yaml` → `<task>.model`):

- **Classification → logistic regression.** It has the best validation accuracy (0.715 vs 0.553 for the dummy, a decisive gap), and train ≈ validation, so it doesn't overfit. The random forest has slightly better F1/ROC-AUC, but it is 0.006 behind on accuracy (not decisive) and overfits (train 0.85). The single depth-5 tree is clearly worse (0.649).
- **Regression → random forest.** It has the lowest validation RMSE (≈ 2008 vs 2104 for ridge and 2472 for the dummy) and the highest R² (≈ 0.30). Its edge over ridge holds on 3 of 5 folds and is not decisive (≈ 96 RMSE, SE 64). It also overfits clearly (train RMSE ≈ 1008), which tuning should address.

Every model beats its dummy floor, so the preprocessing produces usable signal for both tasks. These are **development-set** scores. The locked test set is still untouched and is scored once, at the end, for the final models.

In [ ]:
final_pipelines = {}
for task in ["classification", "regression"]:
    X_dev, _, y_dev, _, _, _ = splits[task]
    final_pipelines[task] = make_pipeline(
        task_preprocessing_config(config, task), TASK_MODELS[task].build_model(config[task]["model"]),
    ).fit(X_dev, y_dev)                                  # all development rows; locked test still untouched
final_pipelines["classification"]

# <font color='#BFD72F' size=6>**11. From this notebook to the pipeline**</font> <a class="anchor" id="to-pipeline"></a>

| This notebook | Pipeline |
|---|---|
| Sections 1–2, 5.1–5.3 | `src/preprocessing.py` → `clean_dataset()` (row-preserving: `standardize_columns`, `flag_invalid_values`, `apply_recompute_rules`, `apply_structural_fill`, `add_flags`, redundant columns); `config.yaml` → `diagnostics` |
| Sections 2.1, 4 | `drop_training_rows()`: training file only, before the split |
| Section 2.4–2.5 | `regression/preprocessing.py` → `select_training_rows()` (`config.yaml` → `regression.row_filter`) |
| Section 6 | `config.yaml` → `preprocessing.imputation`, `mnar_indicator_sources` |
| Section 7 | `split_features_target()`, `split_dev_test()`; `config.yaml` → `data`, `test_set` |
| Section 8 | `build_preprocessor(task_preprocessing_config(config, task))` |
| Section 9 | `src/evaluate.py` → `temporal_folds()`, `compare_encoders_scalers()`, `paired_comparison()`; `config.yaml` → `validation`, `<task>.comparison_model`, `<task>.preprocessing` |
| Section 10 | `src/evaluate.py` → `make_pipeline()`, `cross_validate_pipeline()`; `classification/model.py`, `regression/model.py` → `build_model()`; `config.yaml` → `<task>.candidate_models`, `<task>.model` |
| Everything | `python main.py` runs it end to end for both tasks: preprocessing, the configured model vs the dummy floor in temporal CV, and the final fit on the development rows |

[Back to TOC](#toc)